In [15]:
model = "intfloat/multilingual-e5-large"

import torch.nn.functional as F

from torch import Tensor
from transformers import AutoTokenizer, AutoModel


def average_pool(last_hidden_states: Tensor,
                 attention_mask: Tensor) -> Tensor:
    last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]


# Each input text should start with "query: " or "passage: ", even for non-English texts.
# For tasks other than retrieval, you can simply use the "query: " prefix.
input_texts = ['query: Как стать супер героем',
               'query: Как стать худым',
               "passage: Надо в это верить",
               "passage: Надо мало есть"]

tokenizer = AutoTokenizer.from_pretrained('intfloat/multilingual-e5-large')
model = AutoModel.from_pretrained('intfloat/multilingual-e5-large')

# Tokenize the input texts
batch_dict = tokenizer(input_texts, max_length=512, padding=True, truncation=True, return_tensors='pt')

outputs = model(**batch_dict)
embeddings = average_pool(outputs.last_hidden_state, batch_dict['attention_mask'])

# normalize embeddings
# embeddings = F.normalize(embeddings, p=2, dim=1)
# scores = (embeddings[:2] @ embeddings[2:].T) * 100
# print(scores.tolist())
# print(embeddings)


[[78.31405639648438, 79.35562896728516], [79.55821990966797, 87.92352294921875]]


In [19]:
batch_dict = tokenizer('Я хочу стать богатым', max_length=512, padding=True, truncation=True, return_tensors='pt')

In [31]:
import torch
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F
from qdrant_client import QdrantClient
from qdrant_client.http.models import PointStruct

# Инициализация модели и токенизатора
model_name = "intfloat/multilingual-e5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Функция получения эмбеддингов
def get_embeddings(texts):
    batch = tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**batch)
    embeddings = outputs.last_hidden_state.mean(dim=1)  # Усреднение
    return F.normalize(embeddings, p=2, dim=1)  # Нормализация

emb = get_embeddings('Побежали в город')

In [84]:
import torch
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F

def generate_embeddings(text, device='cpu'):

    model_name = 'Tochka-AI/ruRoPEBert-e5-base-2k'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name, trust_remote_code=True, attn_implementation='sdpa')

    model = model.to(device)
    tokenized = tokenizer(
        text, return_tensors='pt', padding=True, truncation=True
    )
    tokenized = {key: value.to(device) for key, value in tokenized.items()}

    with torch.inference_mode():
        pooled_output = model(**tokenized).pooler_output


    normalized_embeddings = F.normalize(pooled_output, dim=1)
    return normalized_embeddings

text = "Привет, чем занят?"
device = 'cuda' if torch.cuda.is_available() else 'cpu'
embeddings = generate_embeddings(text, device)
# cosine_similarity = embeddings @ embeddings.T
# print("Косинусное сходство:\n", cosine_similarity)

In [88]:
embeddings.tolist()[0]

[0.07959923148155212,
 -0.0037002600729465485,
 0.005835376214236021,
 -0.002181017305701971,
 0.02501397579908371,
 -0.012488413602113724,
 -0.00597516680136323,
 0.010497074574232101,
 -0.012197600677609444,
 0.0009661348885856569,
 0.0007194424979388714,
 -0.024451997131109238,
 0.003362508025020361,
 -0.027354629710316658,
 0.03246629610657692,
 -0.03728734329342842,
 -0.03441503271460533,
 0.00708899786695838,
 -0.03687809035181999,
 0.04184237867593765,
 0.049634162336587906,
 -0.035577625036239624,
 0.007614294998347759,
 0.01008195336908102,
 0.04071488976478577,
 0.03353136032819748,
 0.10028253495693207,
 -0.05802357941865921,
 -0.00897214189171791,
 -0.060037508606910706,
 -0.018947623670101166,
 0.006985013373196125,
 0.04420144483447075,
 -0.022232942283153534,
 -0.024320585653185844,
 0.04307197779417038,
 -0.010380079969763756,
 -0.0293890330940485,
 0.020405277609825134,
 -0.007371662184596062,
 0.003569439984858036,
 0.0019451957195997238,
 0.012647158466279507,
 0.018

In [59]:
outputs = model(**batch)
F.normalize(outputs.last_hidden_state.mean(dim=1), p=2, dim=1)[0]

tensor([ 0.0193,  0.0130, -0.0031,  ...,  0.0203, -0.0115,  0.0141],
       grad_fn=<SelectBackward0>)

In [1]:
import random

# 7 релевантных объектов (из 20)
relevant_docs = {"doc_1", "doc_2", "doc_3", "doc_4", "doc_5", "doc_6", "doc_7"}

# Симулируем топ-10 с нерелевантными объектами
retrieved_docs = ["doc_1", "doc_2", "doc_8", "doc_9", "doc_10", "doc_3", "doc_4", "doc_11", "doc_12", "doc_5"]

# Считаем количество релевантных документов в топ-10
relevant_in_top_10 = sum(1 for doc in retrieved_docs if doc in relevant_docs)

# Recall@10
recall_at_10 = relevant_in_top_10 / len(relevant_docs)

print(f"Recall@10: {recall_at_10:.2f}")


Recall@10: 0.71
